# Notebook 1: Data Prep & Split
Cleans Dataset 1 (Kaggle Clickbait Dataset) + Dataset 2 (`train.csv` only), combines them, and makes the shared stratified 80/10/10 split.

**Outputs:** `headlines_raw.csv`, `headlines_cleaned.csv`, `headlines_train.csv`, `headlines_test.csv`, `label_conflicts_report.csv`

## 0. Setup

In [1]:
!pip install -q ftfy
import os, re
import pandas as pd
import ftfy
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 120)
log = {}   # removal counts for the final summary

## 1. Load the datasets
Set the filenames to match your downloads. If the files aren't in Colab yet, the cell will ask you to upload them.

In [2]:
import csv
DS1_PATH = "dataset1.csv"   # Kaggle Clickbait Dataset (headline, clickbait)
DS2_PATH = "dataset2.csv"      # Clickbait News Detection -> only train.csv (ignore valid.csv / test.csv)

missing = [p for p in (DS1_PATH, DS2_PATH) if not os.path.exists(p)]
if missing:
    from google.colab import files
    print("Please upload:", missing)
    files.upload()

def read_any(path):
    # Try reading normally first with different encodings
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue

    # If standard reading fails due to quoting/formatting issues, try disabling quoting
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc, quoting=csv.QUOTE_NONE)
        except Exception:
            continue

    raise ValueError(f"Could not read {path}")

ds1_raw = read_any(DS1_PATH)
ds2_raw = read_any(DS2_PATH)
print("Dataset 1:", ds1_raw.shape, list(ds1_raw.columns))
print("Dataset 2:", ds2_raw.shape, list(ds2_raw.columns))

Dataset 1: (32000, 2) ['headline', 'clickbait']
Dataset 2: (24871, 4) ['id', 'title', 'text', 'label']


## 2. Dataset 1 — keep `headline` + `clickbait`, confirm 0/1 meaning

In [3]:
ds1 = ds1_raw[["headline", "clickbait"]].rename(columns={"clickbait": "label"}).copy()
ds1["source"] = "ds1"
print(ds1["label"].value_counts(dropna=False))

# Eyeball check: label 1 should look like clickbait, label 0 like normal news
print("\n--- label 0 samples (should be non-clickbait) ---")
print(ds1[ds1.label == 0]["headline"].sample(5, random_state=1).to_string(index=False))
print("\n--- label 1 samples (should be clickbait) ---")
print(ds1[ds1.label == 1]["headline"].sample(5, random_state=1).to_string(index=False))

label
0    16001
1    15999
Name: count, dtype: int64

--- label 0 samples (should be non-clickbait) ---
                  French woman admits to killing her eight infants
          Discovery of the most powerful explosion in the Universe
Pakistan security forces airstrikes kill several Taliban militants
             South Africa defeats Australia in second cricket test
                   Brazilian President meets President of Colombia

--- label 1 samples (should be clickbait) ---
                               11 Tragic Everyday Disappointments
                                  Are You More Stark Or Lannister
                        The 18 Best Moments For '90s Kids In 2015
                    27 Small But Awesome Ways To Be A Good Friend
This Video Beautifully Shows How Difficult It Can Be To Read Lips


## 3. Dataset 2 — keep `title` + `label`, drop `other`, map news→0 / clickbait→1

In [4]:
ds2 = ds2_raw[["title", "label"]].rename(columns={"title": "headline"}).copy()
ds2["label"] = ds2["label"].astype(str).str.strip().str.lower()
print(ds2["label"].value_counts(dropna=False))

n_other = (ds2["label"] == "other").sum()
log["ds2 rows labeled 'other' removed"] = int(n_other)
ds2 = ds2[ds2["label"] != "other"].copy()

ds2["label"] = ds2["label"].map({"news": 0, "clickbait": 1})   # anything unexpected -> NaN (removed later)
ds2["source"] = "ds2"
print("\nAfter dropping 'other':", ds2.shape)

label
news         14650
other         6473
clickbait     3748
Name: count, dtype: int64

After dropping 'other': (18398, 3)


## 4. Combine

In [5]:
df = pd.concat([ds1, ds2], ignore_index=True)
log["rows after combining"] = len(df)
print(df.shape)
df.head()

(50398, 3)


,headline,label,source
0,Should I Get Bings,1,ds1
1,Which TV Female Friend Group Do You Belong In,1,ds1
2,"The New ""Star Wars: The Force Awakens"" Trailer Is Here To Give You Chills",1,ds1
3,"This Vine Of New York On ""Celebrity Big Brother"" Is Fucking Perfect",1,ds1
4,A Couple Did A Stunning Photo Shoot With Their Baby After Learning She Had An Inoperable Brain Tumor,1,ds1


## 4b. Save the RAW combined dataset
Snapshot taken right after combining, **before** text repair, empty/invalid removal, conflict removal, and dedup. Same two columns and same 0/1 labels as the cleaned file, so the two can be compared directly.

In [6]:
raw_df = df[["headline", "label"]].copy()
raw_df.to_csv("headlines_raw.csv", index=False, encoding="utf-8-sig")
print("Saved headlines_raw.csv", raw_df.shape)

Saved headlines_raw.csv (50398, 2)


## 5. Text repair — mojibake, non-breaking spaces, whitespace
`uncurl_quotes=False` keeps the original punctuation (curly quotes stay curly), so only the broken characters get fixed.

In [7]:
def clean_text(s):
    if not isinstance(s, str):
        return s
    s = ftfy.fix_text(s, uncurl_quotes=False, fix_latin_ligatures=False, fix_character_width=False)
    s = s.replace("\xa0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

before = df["headline"].copy()
df["headline"] = df["headline"].apply(clean_text)
changed = (before.fillna("") != df["headline"].fillna(""))
log["headlines modified by text repair"] = int(changed.sum())

print("Examples of repaired headlines:")
print(pd.DataFrame({"before": before[changed], "after": df.loc[changed, "headline"]}).head(8).to_string())

# Residual mojibake check
resid = df["headline"].astype(str).str.contains(r"â€|Ã|â\x80|Â", regex=True)
print("\nRows still showing mojibake patterns:", int(resid.sum()))

Examples of repaired headlines:
                                                                                                     before                                                                                                after
2563                                                  This  Beautifully Inclusive Ad Will Make You Ugly Cry                                                 This Beautifully Inclusive Ad Will Make You Ugly Cry
3109                A Photographer Took Pics Of Cats From Underneath To Show Us A Side Of Them Rarely  Seen               A Photographer Took Pics Of Cats From Underneath To Show Us A Side Of Them Rarely Seen
5998                                          24 Words Only  "How I Met Your Mother" Fans Really Understand                                         24 Words Only "How I Met Your Mother" Fans Really Understand
6339                                                        18 Pics That'll Make You  Miss Your Best Friend                         

## 6. Remove missing/empty headlines, invalid labels

In [8]:
n0 = len(df)
df["headline"] = df["headline"].replace("", pd.NA)
df = df.dropna(subset=["headline"])
log["missing/empty headlines removed"] = n0 - len(df)

n0 = len(df)
df = df[df["label"].isin([0, 1])].copy()
df["label"] = df["label"].astype(int)
log["invalid labels removed"] = n0 - len(df)
print(df.shape)

(50398, 3)


## 7. Conflicting labels + exact duplicates
Conflicts = same headline with both labels. They're reported to a CSV and removed entirely (no arbitrary picking).

In [9]:
n_labels = df.groupby("headline")["label"].nunique()
conflict_heads = n_labels[n_labels > 1].index

conflicts = df[df["headline"].isin(conflict_heads)].sort_values("headline")
conflicts.to_csv("label_conflicts_report.csv", index=False, encoding="utf-8-sig")
log["unique headlines w/ conflicting labels"] = len(conflict_heads)
log["rows removed for conflicting labels"] = len(conflicts)
print(f"{len(conflict_heads)} conflicting headlines ({len(conflicts)} rows)")
print(conflicts.head(10).to_string())

df = df[~df["headline"].isin(conflict_heads)]

n0 = len(df)
df = df.drop_duplicates(subset="headline", keep="first")
log["exact duplicate headlines removed"] = n0 - len(df)

19 conflicting headlines (61 rows)
                                                                   headline  label source
33725  14 Things We Learned From Caitlyn Jenner’s ‘20/20’ Interview [Video]      1    ds2
39056  14 Things We Learned From Caitlyn Jenner’s ‘20/20’ Interview [Video]      0    ds2
50367                    7 Conservative Classics Every American Should Read      0    ds2
37410                    7 Conservative Classics Every American Should Read      1    ds2
35990                            As Trump Makes Threats, Iran Makes Friends      1    ds2
40686                            As Trump Makes Threats, Iran Makes Friends      0    ds2
38246                                                          BBC Trending      1    ds2
38031                                                          BBC Trending      1    ds2
39364                                                          BBC Trending      0    ds2
37838                                                          BB

## 8. Review the final result

In [10]:
log["FINAL row count"] = len(df)
print(pd.Series(log).to_string())

print("\nLabel counts:")
print(df["label"].value_counts().sort_index().rename({0: "0 (non-clickbait)", 1: "1 (clickbait)"}))
print("\nLabel share:")
print(df["label"].value_counts(normalize=True).round(3))
print("\nRows per source:")
print(df.groupby("source")["label"].value_counts().unstack(fill_value=0))

df = df.reset_index(drop=True)
print("\nRandom samples:")
df.sample(10, random_state=7)[["headline", "label", "source"]]

ds2 rows labeled 'other' removed           6473
rows after combining                      50398
headlines modified by text repair          3306
missing/empty headlines removed               0
invalid labels removed                        0
unique headlines w/ conflicting labels       19
rows removed for conflicting labels          61
exact duplicate headlines removed            76
FINAL row count                           50261

Label counts:
label
0 (non-clickbait)    30558
1 (clickbait)        19703
Name: count, dtype: int64

Label share:
label
0    0.608
1    0.392
Name: proportion, dtype: float64

Rows per source:
label       0      1
source              
ds1     16001  15998
ds2     14557   3705

Random samples:


,headline,label,source
31095,"Even at Carolina, Penguins Dominate Hurricanes",0,ds1
16359,United States re-elects Barack Obama,0,ds1
19946,Sebastian Vettel wins 2009 Japanese Grand Prix,0,ds1
25407,Quake-hit Haiti struck by Hurricane Tomas,0,ds1
34672,Trump's Attorney General Pick Opposes Hate Crime Laws And Roe v. Wade,0,ds2
32471,Ingenious Hack Turns Amazon Dash Button Into Trump-Fighting Machine,1,ds2
35312,A Gaza ‘Tunnel Millionaire’ Falls on Hard Times,1,ds2
30789,Re-evaluation of National Security Ordered,0,ds1
41858,"Trump, Bannon call media the 'opposition party': Does that go too far?",0,ds2
2617,26 State Fair Foods From 2015 That Will Blow Your Mind,1,ds1


## 9. Save cleaned dataset (UTF-8 with BOM so Excel shows text correctly)

In [11]:
df[["headline", "label"]].to_csv("headlines_cleaned.csv", index=False, encoding="utf-8-sig")
print("Saved headlines_cleaned.csv", df.shape)

Saved headlines_cleaned.csv (50261, 3)


## 10. Shared stratified 80/10/10 split
Fixed `random_state` so everyone gets identical files. Both models (TF-IDF+LR and DistilBERT) must use these same two files.

In [12]:
clean = pd.read_csv("headlines_cleaned.csv", encoding="utf-8-sig")

# 80/20 split, 20% will be for both test and validation sets
train_df, validation_test_temp_df = train_test_split(
    clean, test_size=0.2, stratify=clean["label"], random_state=42, shuffle=True
)

# Split 20% of data into half for the test and validation sets
test_df, val_df = train_test_split(
    validation_test_temp_df, test_size=0.5, stratify=validation_test_temp_df["label"], random_state=42, shuffle=True
)

train_df.to_csv("headlines_train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv("headlines_validation.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("headlines_test.csv", index=False, encoding="utf-8-sig")

print("Train:", train_df.shape, "| Validation:", val_df.shape, "| Test:", test_df.shape)
for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name} label share:\n", d["label"].value_counts(normalize=True).round(4))

assert not set(train_df["headline"]) & set(val_df["headline"]), "Leakage: train/validation overlap!"
assert not set(train_df["headline"]) & set(test_df["headline"]), "Leakage: train/test overlap!"
assert not set(val_df["headline"]) & set(test_df["headline"]), "Leakage: validation/test overlap!"
print("\nNo headline overlap between train, validation, and test ✔")

Train: (40208, 2) | Validation: (5027, 2) | Test: (5026, 2)

Train label share:
 label
0    0.608
1    0.392
Name: proportion, dtype: float64

Validation label share:
 label
0    0.6079
1    0.3921
Name: proportion, dtype: float64

Test label share:
 label
0    0.608
1    0.392
Name: proportion, dtype: float64

No headline overlap between train, validation, and test ✔


## 11. Optional, download files to check .csv files

In [13]:
try:
    from google.colab import files
    for f in ["headlines_raw.csv", "headlines_cleaned.csv", "headlines_train.csv",
              "headlines_validation.csv", "headlines_test.csv", "label_conflicts_report.csv"]:
        files.download(f)
except ImportError:
    print("Not in Colab - files are in the working directory.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (2 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>